# Notebook 02 — Cellpose Segmentation

Run Cellpose on the Xenium DAPI nuclear staining image to produce nucleus segmentation masks. 
We run a parameter sweep over cell diameter to assess robustness, then export polygon boundaries for downstream comparison.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from pathlib import Path
from cellpose import models, plot as cp_plot
from skimage import measure
from shapely.geometry import Polygon
import json

from src.visualization.spatial_plots import plot_cell_boundaries
from src.evaluation.metrics import transcript_assignment_stats, parameter_sensitivity

DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

## 1. Load DAPI Image

In [ ]:
dapi = tifffile.imread(DATA_DIR / "morphology.ome.tif")
print(f"DAPI image shape: {dapi.shape}, dtype: {dapi.dtype}")

# Normalise for display
dapi_norm = (dapi - dapi.min()) / (dapi.max() - dapi.min())

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(dapi_norm, cmap="gray", origin="lower")
ax.set_title("DAPI Nuclear Staining")
ax.axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "dapi_image.png", dpi=150)
plt.show()

## 2. Run Cellpose (Default Diameter)

In [ ]:
model = models.Cellpose(gpu=False, model_type="nuclei")

DEFAULT_DIAMETER = 30  # pixels; estimated from DAPI image scale

masks, flows, styles, diams = model.eval(
    dapi,
    diameter=DEFAULT_DIAMETER,
    channels=[0, 0],
    flow_threshold=0.4,
    cellprob_threshold=0.0,
)

n_cells = masks.max()
print(f"Cellpose detected {n_cells:,} nuclei (diameter={DEFAULT_DIAMETER}px)")

## 3. Visualise Segmentation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].imshow(dapi_norm, cmap="gray", origin="lower")
axes[0].set_title("DAPI (input)")
axes[0].axis("off")

axes[1].imshow(dapi_norm, cmap="gray", origin="lower")
axes[1].imshow(masks, cmap="nipy_spectral", alpha=0.4, origin="lower")
axes[1].set_title(f"Cellpose Segmentation ({n_cells:,} nuclei)")
axes[1].axis("off")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "cellpose_segmentation_map.png", dpi=150)
plt.show()

## 4. Extract Polygon Boundaries

In [ ]:
def masks_to_polygons(mask_array: np.ndarray) -> list:
    """Convert an integer label mask to a list of Shapely Polygons."""
    polys = []
    for region in measure.regionprops(mask_array):
        contours = measure.find_contours(mask_array == region.label, 0.5)
        if not contours:
            continue
        # Use longest contour
        contour = max(contours, key=len)
        poly = Polygon(contour[:, [1, 0]])  # (col, row) -> (x, y)
        if poly.is_valid and poly.area > 0:
            polys.append(poly)
    return polys


cellpose_polys = masks_to_polygons(masks)
print(f"Extracted {len(cellpose_polys):,} valid polygon boundaries")

# Cell area distribution
areas = [p.area for p in cellpose_polys]
print(f"Cell area (px²) — mean: {np.mean(areas):.1f}, median: {np.median(areas):.1f}, std: {np.std(areas):.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(areas, bins=60, color="steelblue", edgecolor="none")
ax.set_xlabel("Cell area (px²)")
ax.set_ylabel("Count")
ax.set_title("Cellpose — Cell Area Distribution")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "cellpose_area_distribution.png", dpi=150)
plt.show()

## 5. Assign Transcripts to Cellpose Cells

Point-in-polygon assignment: each transcript is labelled with the mask value at its pixel coordinate.

In [ ]:
transcripts = pd.read_parquet(DATA_DIR / "transcripts_filtered.parquet")

# Map µm coordinates to pixel coordinates
# Xenium provides pixel_size (µm/px) in the experiment.xenium JSON
PIXEL_SIZE = 0.2125  # µm per pixel (standard Xenium)

transcripts["col_px"] = (transcripts["x_location"] / PIXEL_SIZE).astype(int)
transcripts["row_px"] = (transcripts["y_location"] / PIXEL_SIZE).astype(int)

# Clip to image bounds
h, w = masks.shape
transcripts["col_px"] = transcripts["col_px"].clip(0, w - 1)
transcripts["row_px"] = transcripts["row_px"].clip(0, h - 1)

transcripts["cellpose_cell_id"] = masks[
    transcripts["row_px"].values,
    transcripts["col_px"].values,
]

stats = transcript_assignment_stats(transcripts, cell_col="cellpose_cell_id")
print("Cellpose transcript assignment stats:")
for k, v in stats.items():
    print(f"  {k}: {v}")

## 6. Parameter Sweep — Diameter Sensitivity

In [ ]:
diameters = [15, 20, 25, 30, 35, 40, 50]
sweep_results = []

for diam in diameters:
    m, _, _, _ = model.eval(dapi, diameter=diam, channels=[0, 0])
    n = int(m.max())
    sweep_results.append({"diameter": diam, "n_cells": n})
    print(f"diameter={diam}px → {n:,} cells")

sweep_df = parameter_sensitivity(sweep_results, param_name="diameter", metric="n_cells")
print(f"\nCoefficient of variation (cell count): {sweep_df['cv'].iloc[0]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sweep_df["diameter"], sweep_df["n_cells"], marker="o", color="steelblue")
ax.axvline(DEFAULT_DIAMETER, color="red", linestyle="--", label=f"Default ({DEFAULT_DIAMETER}px)")
ax.set_xlabel("Diameter (px)")
ax.set_ylabel("Cells detected")
ax.set_title("Cellpose — Parameter Sensitivity (Diameter)")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "cellpose_diameter_sweep.png", dpi=150)
plt.show()

## 7. Save Outputs

In [ ]:
np.save(DATA_DIR / "cellpose_masks.npy", masks)
transcripts.to_parquet(DATA_DIR / "transcripts_cellpose.parquet", index=False)
sweep_df.to_csv(RESULTS_DIR / "cellpose_diameter_sweep.csv", index=False)
print("Saved Cellpose masks and transcript assignments.")